# Pairwise sentence SMATCH for one Liputan 6 document

Set `DOCUMENT_ID` and `SPLIT`, then use **Run All** inside this notebook. The notebook shows every unique pair of sentences in the document beside its SMATCH score. No terminal command is required.

In [ ]:
from __future__ import annotations
import importlib.util
import re
import zipfile
from pathlib import Path, PurePosixPath
import pandas as pd
from IPython.display import display

## Input

Enter the document ID without a sentence suffix or `.txt`.

In [ ]:
DOCUMENT_ID = '26652'
SPLIT = 'train'  # 'train' or 'test'

In [ ]:
def find_project_root():
    # Works when Jupyter opens in the repository or any subdirectory.
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'scripts' / 'liputan6' / 'build_smatch_adjacency.py').is_file():
            return candidate
    # Local fallback lets this notebook run when opened directly from the UI.
    local_project = Path(r'D:\Github\generate_amr')
    if (local_project / 'scripts' / 'liputan6' / 'build_smatch_adjacency.py').is_file():
        return local_project
    raise RuntimeError('Could not locate the generate_amr project folder.')

PROJECT_ROOT = find_project_root()
builder_path = PROJECT_ROOT / 'scripts' / 'liputan6' / 'build_smatch_adjacency.py'
spec = importlib.util.spec_from_file_location('build_smatch_adjacency', builder_path)
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)
if SPLIT not in {'train', 'test'}:
    raise ValueError("SPLIT must be 'train' or 'test'")
DOCUMENT_ID = str(DOCUMENT_ID).strip()
if not DOCUMENT_ID:
    raise ValueError('DOCUMENT_ID cannot be empty')

In [ ]:
SENTENCE_RE = re.compile(r'^# ::snt\s+(.*)$', re.MULTILINE)
MEMBER_RE = re.compile(r'^(?P<doc_id>.+)_(?P<sent_idx>\d+)\.txt$')

def sentence_text(amr_text):
    match = SENTENCE_RE.search(amr_text)
    return match.group(1).strip() if match else '[sentence metadata unavailable]'

def load_document(doc_id, split):
    data_dir = PROJECT_ROOT / 'data' / 'liputan6' / 'amr_graphs'
    extracted_dir = data_dir / 'extracted' / split / 'amr_graphs'
    records = []
    if extracted_dir.is_dir():
        for path in extracted_dir.glob(f'{doc_id}_*.txt'):
            match = MEMBER_RE.fullmatch(path.name)
            if match and match.group('doc_id') == doc_id:
                records.append((int(match.group('sent_idx')), path.read_text(encoding='utf-8-sig')))
    else:
        archive_path = data_dir / 'archives' / f'{split}.zip'
        with zipfile.ZipFile(archive_path) as archive:
            for info in archive.infolist():
                name = PurePosixPath(info.filename).name
                match = MEMBER_RE.fullmatch(name)
                if match and match.group('doc_id') == doc_id:
                    records.append((int(match.group('sent_idx')), archive.read(info).decode('utf-8-sig')))
    if not records:
        raise ValueError(f'Document {doc_id!r} was not found in the {split} split')
    result = []
    for sent_idx, content in sorted(records):
        result.append({'sent_idx': sent_idx, 'sentence': sentence_text(content),
                       'amr': builder.parse_one_graph(content, f'{doc_id}_{sent_idx}.txt')})
    return result

In [ ]:
sentences = load_document(DOCUMENT_ID, SPLIT)
print(f'Document {DOCUMENT_ID} ({SPLIT}): {len(sentences)} sentences')
display(pd.DataFrame(sentences)[['sent_idx', 'sentence']])

In [ ]:
pairs = []
for i, first in enumerate(sentences):
    for second in sentences[i + 1:]:
        pairs.append({
            'sentence_a_idx': first['sent_idx'],
            'sentence_a': first['sentence'],
            'sentence_b_idx': second['sent_idx'],
            'sentence_b': second['sentence'],
            'smatch_score': builder.smatch_details(first['amr'], second['amr'])['f_score'],
        })
pairwise_scores = (
    pd.DataFrame(pairs)
    .sort_values('smatch_score', ascending=False)
    .reset_index(drop=True)
)
if pairwise_scores.empty:
    print('This document has one sentence, so it has no sentence pairs.')
else:
    display(pairwise_scores.style.format({'smatch_score': '{:.4f}'}))

## Inspect matching triples for one pair

Choose two sentence indices from the table above. This displays the exact aligned triples counted by SMATCH and the resulting precision, recall, and F-score.

In [ ]:
FIRST_SENTENCE_IDX = 2
SECOND_SENTENCE_IDX = 7

sentence_by_idx = {item['sent_idx']: item for item in sentences}
if FIRST_SENTENCE_IDX not in sentence_by_idx or SECOND_SENTENCE_IDX not in sentence_by_idx:
    raise ValueError('Both selected sentence indices must exist in the sentence table above.')
if FIRST_SENTENCE_IDX == SECOND_SENTENCE_IDX:
    raise ValueError('Select two different sentence indices.')

first_selected = sentence_by_idx[FIRST_SENTENCE_IDX]
second_selected = sentence_by_idx[SECOND_SENTENCE_IDX]
details = builder.smatch_details(first_selected['amr'], second_selected['amr'])

print(f"Sentence {FIRST_SENTENCE_IDX}: {first_selected['sentence']}")
print(f"Sentence {SECOND_SENTENCE_IDX}: {second_selected['sentence']}")
print()
print(f"AMR graph for sentence {FIRST_SENTENCE_IDX}:")
print(builder.penman.encode(builder.penman.decode(first_selected['amr']), indent=4))
print()
print(f"AMR graph for sentence {SECOND_SENTENCE_IDX}:")
print(builder.penman.encode(builder.penman.decode(second_selected['amr']), indent=4))
print()
print(f"Matching triples: {details['match_count']}")
print(f"First graph triples: {details['first_count']}")
print(f"Second graph triples: {details['second_count']}")
print(f"Precision: {details['precision']:.4f}")
print(f"Recall: {details['recall']:.4f}")
print(f"SMATCH F-score: {details['f_score']:.4f}")

matching_triples = pd.DataFrame(details['matching_triples'])
display(matching_triples)

## Optional adjacency-matrix view

The same scores arranged as a symmetric matrix. The diagonal is zero, matching the saved `.npy` files.

In [ ]:
labels = [f"sentence_{item['sent_idx']}" for item in sentences]
matrix = pd.DataFrame(0.0, index=labels, columns=labels)
for pair in pairs:
    first_label = f"sentence_{pair['sentence_a_idx']}"
    second_label = f"sentence_{pair['sentence_b_idx']}"
    matrix.loc[first_label, second_label] = pair['smatch_score']
    matrix.loc[second_label, first_label] = pair['smatch_score']
display(matrix.style.format('{:.4f}'))